# Sitzung 7: Hilfsmittelfreier Teil — Interaktiv trainieren

In diesem Notebook kannst du:
1. **⏱ Prüfungssimulation** mit Countdown-Timer (40 Min)
2. **Graphen zuordnen** (f, f', f'') und dein Verständnis prüfen
3. **Ableitungen** im Quiz-Modus üben — sympy kontrolliert
4. **Stammfunktionen** analog trainieren
5. **Bestimmte Integrale** im Kopf berechnen und prüfen
6. **Tangentengleichung** aufstellen üben
7. **Extrema Schritt für Schritt** bestimmen
8. **Begründungsaufgaben** zu notwendiger/hinreichender Bedingung aktiv formulieren
9. **Typische Fehler** kennen und vermeiden

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from ipywidgets import interact, Button, Text, Textarea, Output, VBox, HBox, Dropdown, HTML, Label
from IPython.display import display, Markdown, clear_output
import random
import threading
import time as _time

%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 13
plt.rcParams['text.usetex'] = False
plt.rcParams['mathtext.fontset'] = 'cm'  # Computer Modern (LaTeX-Look)

x = sp.Symbol('x')

---
## Prüfungs-Timer (40 Minuten)

Starte den Timer, wenn du den Diagnosetest als Prüfungssimulation bearbeitest.
Der Timer zählt von 40:00 herunter — genau wie im echten Abitur.

In [ ]:
def pruefungs_timer(minuten=40):
    """Countdown-Timer für die Prüfungssimulation."""
    timer_html = HTML(
        value=f'<h2 style="font-family: monospace; color: #2ca02c;">⏱ {minuten:02d}:00</h2>',
        layout={'width': '250px'},
    )
    btn_start = Button(description='Timer starten', button_style='success', icon='play')
    btn_stop = Button(description='Stopp', button_style='danger', icon='stop')
    btn_stop.disabled = True
    status = {'running': False}

    def countdown():
        total = minuten * 60
        while total > 0 and status['running']:
            mins, secs = divmod(total, 60)
            if total <= 300:  # letzte 5 Minuten: rot
                farbe = '#d62728'
            elif total <= 600:  # letzte 10 Minuten: orange
                farbe = '#ff7f0e'
            else:
                farbe = '#2ca02c'
            timer_html.value = (
                f'<h2 style="font-family: monospace; color: {farbe};">'
                f'⏱ {mins:02d}:{secs:02d}</h2>'
            )
            _time.sleep(1)
            total -= 1
        if total <= 0 and status['running']:
            timer_html.value = (
                '<h2 style="font-family: monospace; color: #d62728;">'
                '⏱ 00:00 — Zeit abgelaufen!</h2>'
            )
            status['running'] = False
            btn_start.disabled = False
            btn_stop.disabled = True

    def start(b):
        status['running'] = True
        btn_start.disabled = True
        btn_stop.disabled = False
        t = threading.Thread(target=countdown, daemon=True)
        t.start()

    def stop(b):
        status['running'] = False
        btn_start.disabled = False
        btn_stop.disabled = True

    btn_start.on_click(start)
    btn_stop.on_click(stop)
    display(VBox([timer_html, HBox([btn_start, btn_stop])]))

pruefungs_timer()

---
## 1. Graphen-Zuordnung: f, f' und f''

Drei Graphen werden angezeigt — einer gehört zu f, einer zu f' und einer zu f''.
Wähle die richtige Zuordnung und klicke auf **Auflösen**.

In [ ]:
# Graphen-Zuordnungs-Aufgaben: Sammlung
graphen_aufgaben = [
    {
        'f': lambda t: t**3 - 3*t,
        'fp': lambda t: 3*t**2 - 3,
        'fpp': lambda t: 6*t,
        'f_latex': r'$f(x) = x^3 - 3x$',
        'fp_latex': r"$f'(x) = 3x^2 - 3$",
        'fpp_latex': r"$f''(x) = 6x$",
    },
    {
        'f': lambda t: -t**4 + 4*t**2,
        'fp': lambda t: -4*t**3 + 8*t,
        'fpp': lambda t: -12*t**2 + 8,
        'f_latex': r'$f(x) = -x^4 + 4x^2$',
        'fp_latex': r"$f'(x) = -4x^3 + 8x$",
        'fpp_latex': r"$f''(x) = -12x^2 + 8$",
    },
    {
        'f': lambda t: np.sin(t),
        'fp': lambda t: np.cos(t),
        'fpp': lambda t: -np.sin(t),
        'f_latex': r'$f(x) = \sin(x)$',
        'fp_latex': r"$f'(x) = \cos(x)$",
        'fpp_latex': r"$f''(x) = -\sin(x)$",
    },
    {
        'f': lambda t: t**4/4 - 2*t**2 + 1,
        'fp': lambda t: t**3 - 4*t,
        'fpp': lambda t: 3*t**2 - 4,
        'f_latex': r'$f(x) = \tfrac{1}{4}x^4 - 2x^2 + 1$',
        'fp_latex': r"$f'(x) = x^3 - 4x$",
        'fpp_latex': r"$f''(x) = 3x^2 - 4$",
    },
]

def graphen_zuordnung():
    aufgabe = random.choice(graphen_aufgaben)
    xs = np.linspace(-3, 3, 400)

    kurven = [
        ('f', aufgabe['f'], aufgabe['f_latex']),
        ("f'", aufgabe['fp'], aufgabe['fp_latex']),
        ("f''", aufgabe['fpp'], aufgabe['fpp_latex']),
    ]
    # Zufällige Reihenfolge der Anzeige
    anzeige_reihenfolge = list(range(3))
    random.shuffle(anzeige_reihenfolge)
    labels_anzeige = ['Graph 1', 'Graph 2', 'Graph 3']
    farben = ['#1f77b4', '#d62728', '#2ca02c']

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    for i, idx in enumerate(anzeige_reihenfolge):
        _, fn, _ = kurven[idx]
        ys = fn(xs)
        axes[i].plot(xs, ys, color=farben[i], linewidth=2.5)
        axes[i].axhline(y=0, color='k', linewidth=0.5)
        axes[i].axvline(x=0, color='k', linewidth=0.5)
        axes[i].grid(True, alpha=0.3)
        axes[i].set_title(labels_anzeige[i], fontsize=14)
        axes[i].set_xlim(-3, 3)
        y_range = np.nanmax(np.abs(ys[np.isfinite(ys)]))
        axes[i].set_ylim(-max(y_range * 1.3, 1), max(y_range * 1.3, 1))
    plt.tight_layout()
    plt.show()

    # Lösung speichern
    loesung = {}
    for i, idx in enumerate(anzeige_reihenfolge):
        name, _, latex = kurven[idx]
        loesung[labels_anzeige[i]] = (name, latex)

    # Dropdowns für Zuordnung
    dd_f = Dropdown(options=['Graph 1', 'Graph 2', 'Graph 3'], description='f =')
    dd_fp = Dropdown(options=['Graph 1', 'Graph 2', 'Graph 3'], value='Graph 2', description="f' =")
    dd_fpp = Dropdown(options=['Graph 1', 'Graph 2', 'Graph 3'], value='Graph 3', description="f'' =")
    btn = Button(description='Auflösen', button_style='info')
    out = Output()

    def aufloesung(b):
        with out:
            clear_output()
            # Prüfe Antworten
            antwort_richtig = True
            for dd, rolle in [(dd_f, 'f'), (dd_fp, "f'"), (dd_fpp, "f''")]:
                graph_name = dd.value
                tatsaechlich = loesung[graph_name][0]
                if tatsaechlich != rolle:
                    antwort_richtig = False

            if antwort_richtig:
                display(Markdown("### Richtig!"))
            else:
                display(Markdown("### Nicht ganz. Hier die Auflösung:"))

            for graph_name in labels_anzeige:
                rolle, latex = loesung[graph_name]
                display(Markdown(f"- **{graph_name}** = **{rolle}**: {latex}"))

            display(Markdown("\n**Tipp:** Nullstellen von f' = Extrema von f. "
                             "Nullstellen von f'' = Wendepunkte von f."))

    btn.on_click(aufloesung)
    display(VBox([HBox([dd_f, dd_fp, dd_fpp]), btn, out]))

graphen_zuordnung()

**Tipp:** Führe die Zelle mehrfach aus (Shift+Enter), um neue Aufgaben zu bekommen!

---
## 2. Ableitungs-Quiz

Eine zufällige Funktion wird angezeigt. Gib die Ableitung ein und lass sie von sympy prüfen.

In [ ]:
# Hilfsfunktion: sympy-Ausdruck sicher parsen
def parse_eingabe(text):
    """Parst eine Benutzereingabe zu einem sympy-Ausdruck."""
    text = text.strip().replace('^', '**')
    return sp.sympify(
        text,
        locals={
            'x': x, 'e': sp.E, 'ln': sp.log, 'log': sp.log,
            'exp': sp.exp, 'sqrt': sp.sqrt, 'sin': sp.sin,
            'cos': sp.cos, 'tan': sp.tan, 'pi': sp.pi,
        },
    )

# Aufgabenpool Ableitungen
ableitung_pool = [
    x**3 - 5*x**2 + 2*x - 1,
    4*x**4 - x**2 + 3,
    sp.exp(x) * x**2,
    sp.sin(x) * x,
    sp.log(x**2 + 1),
    (2*x + 1)**5,
    sp.exp(-3*x),
    x * sp.cos(x),
    sp.sqrt(x**2 + 4),
    3*x**2 * sp.exp(x),
    sp.sin(x**2),
    1 / (x**2 + 1),
    x**3 * sp.log(x),
    sp.exp(x) * sp.sin(x),
    (x**2 - 1)**3,
]

def ableitungs_quiz():
    aufgabe = random.choice(ableitung_pool)
    loesung = sp.diff(aufgabe, x)

    display(Markdown(f"### Bestimme die Ableitung von:"))
    display(Markdown(f"$$f(x) = {sp.latex(aufgabe)}$$"))

    eingabe = Text(
        placeholder='z.B. 3*x^2 - 10*x + 2',
        description="f'(x) =",
        style={'description_width': '60px'},
        layout={'width': '500px'},
        continuous_update=False,
    )
    btn = Button(description='Prüfen', button_style='success')
    btn_loesung = Button(description='Lösung zeigen', button_style='warning')
    out = Output()

    def pruefen(b):
        with out:
            clear_output()
            try:
                antwort = parse_eingabe(eingabe.value)
                diff = sp.simplify(antwort - loesung)
                if diff == 0:
                    display(Markdown("### Richtig!"))
                else:
                    display(Markdown("### Leider falsch."))
                    display(Markdown(f"Deine Antwort: $f'(x) = {sp.latex(antwort)}$"))
                    display(Markdown(f"Richtig wäre: $f'(x) = {sp.latex(sp.simplify(loesung))}$"))
            except Exception as e:
                display(Markdown(f"**Eingabefehler:** {e}"))

    def zeige_loesung(b):
        with out:
            clear_output()
            display(Markdown(f"**Lösung:** $f'(x) = {sp.latex(sp.simplify(loesung))}$"))

    btn.on_click(pruefen)
    btn_loesung.on_click(zeige_loesung)
    display(VBox([eingabe, HBox([btn, btn_loesung]), out]))

ableitungs_quiz()

**Eingabeformat:** `x^2` oder `x**2`, `e^x` oder `exp(x)`, `ln(x)`, `sin(x)`, `cos(x)`, `sqrt(x)`

Führe die Zelle erneut aus für eine neue Aufgabe!

---
## 3. Stammfunktions-Quiz

Analog zum Ableitungs-Quiz — diesmal umgekehrt: Finde die Stammfunktion F(x).

In [ ]:
# Aufgabenpool Stammfunktionen
stammfunktion_pool = [
    3*x**2 + 2*x - 1,
    6*x**5 - 4*x**3,
    sp.exp(2*x),
    sp.cos(x),
    sp.sin(3*x),
    4*x**3 - 6*x,
    1/x,
    2*sp.exp(x) + 3,
    x**(-2),
    5*x**4,
    sp.cos(2*x),
    10*x - 3,
]

def stammfunktions_quiz():
    aufgabe = random.choice(stammfunktion_pool)
    loesung = sp.integrate(aufgabe, x)

    display(Markdown(f"### Bestimme eine Stammfunktion von:"))
    display(Markdown(f"$$f(x) = {sp.latex(aufgabe)}$$"))
    display(Markdown("*(Integrationskonstante C weglassen)*"))

    eingabe = Text(
        placeholder='z.B. x^3 + x^2 - x',
        description="F(x) =",
        style={'description_width': '60px'},
        layout={'width': '500px'},
        continuous_update=False,
    )
    btn = Button(description='Prüfen', button_style='success')
    btn_loesung = Button(description='Lösung zeigen', button_style='warning')
    out = Output()

    def pruefen(b):
        with out:
            clear_output()
            try:
                antwort = parse_eingabe(eingabe.value)
                # Prüfung: Ableitung der Antwort muss f ergeben
                abl_antwort = sp.diff(antwort, x)
                diff = sp.simplify(abl_antwort - aufgabe)
                if diff == 0:
                    display(Markdown("### Richtig!"))
                else:
                    display(Markdown("### Leider falsch."))
                    display(Markdown(f"Deine Antwort: $F(x) = {sp.latex(antwort)}$"))
                    display(Markdown(f"Ableitung davon: $F'(x) = {sp.latex(sp.simplify(abl_antwort))}$ "
                                     f"(sollte $= {sp.latex(aufgabe)}$ sein)"))
                    display(Markdown(f"Richtig wäre z.B.: $F(x) = {sp.latex(loesung)}$"))
            except Exception as e:
                display(Markdown(f"**Eingabefehler:** {e}"))

    def zeige_loesung(b):
        with out:
            clear_output()
            display(Markdown(f"**Lösung:** $F(x) = {sp.latex(loesung)}$"))

    btn.on_click(pruefen)
    btn_loesung.on_click(zeige_loesung)
    display(VBox([eingabe, HBox([btn, btn_loesung]), out]))

stammfunktions_quiz()

---
## 4. Schnell-Rechner: Bestimmte Integrale im Kopf

Berechne das bestimmte Integral im Kopf und gib das Ergebnis als Zahl ein.

In [ ]:
# Aufgabenpool bestimmte Integrale (mit ganzzahligen/einfachen Grenzen)
integral_pool = [
    (2*x + 1, 0, 3),
    (x**2, 0, 2),
    (x**2, 1, 2),
    (3*x**2 - 2*x, 0, 1),
    (x**3, 0, 2),
    (sp.exp(x), 0, 1),
    (6*x**2 + 2, -1, 1),
    (4*x - 1, 0, 2),
    (x**2 - 4, 0, 2),
    (1, 0, 5),
    (2*x, -1, 3),
    (x**2 + x, 0, 2),
]

def integral_quiz():
    f_expr, a, b = random.choice(integral_pool)
    loesung = sp.integrate(f_expr, (x, a, b))

    display(Markdown(f"### Berechne im Kopf:"))
    display(Markdown(f"$$\\int_{{{a}}}^{{{b}}} \\left({sp.latex(f_expr)}\\right) \\, dx$$"))

    eingabe = Text(
        placeholder='z.B. 12 oder 3/2 oder e-1',
        description="Ergebnis =",
        style={'description_width': '80px'},
        layout={'width': '400px'},
        continuous_update=False,
    )
    btn = Button(description='Prüfen', button_style='success')
    btn_loesung = Button(description='Lösung zeigen', button_style='warning')
    out = Output()

    def pruefen(b_click):
        with out:
            clear_output()
            try:
                antwort = parse_eingabe(eingabe.value)
                diff = sp.simplify(antwort - loesung)
                if diff == 0:
                    display(Markdown("### Richtig!"))
                    display(Markdown(f"$$\\int_{{{a}}}^{{{b}}} \\left({sp.latex(f_expr)}\\right) dx "
                                     f"= {sp.latex(loesung)}$$"))
                else:
                    display(Markdown("### Leider falsch."))
                    display(Markdown(f"Deine Antwort: ${sp.latex(antwort)}$"))
                    display(Markdown(f"Richtig: ${sp.latex(loesung)}$"))
            except Exception as e:
                display(Markdown(f"**Eingabefehler:** {e}"))

    def zeige_loesung(b_click):
        with out:
            clear_output()
            F_expr = sp.integrate(f_expr, x)
            display(Markdown(f"**Rechenweg:**"))
            display(Markdown(f"$$F(x) = {sp.latex(F_expr)}$$"))
            display(Markdown(f"$$F({b}) - F({a}) = {sp.latex(F_expr.subs(x, b))} - "
                             f"{sp.latex(F_expr.subs(x, a))} = {sp.latex(loesung)}$$"))

    btn.on_click(pruefen)
    btn_loesung.on_click(zeige_loesung)
    display(VBox([eingabe, HBox([btn, btn_loesung]), out]))

integral_quiz()

**Tipp:** Brüche als `3/2`, Euler-Zahl als `e` eingeben (z.B. `e - 1`).

---
## 8. Typische Begründungsaufgaben: Notwendige und hinreichende Bedingung

**Wichtig:** Schreibe zuerst deine eigene Begründung ins Textfeld, bevor du die Lösung aufdeckst!
So trainierst du, wie im Abitur zu formulieren.

---
## 6. Tangentengleichung aufstellen

Bestimme die Tangentengleichung an f im gegebenen Punkt.
Erinnerung: $t(x) = f(x_0) + f'(x_0) \cdot (x - x_0)$

In [ ]:
# Aufgabenpool Tangenten
tangenten_pool = [
    (x**3 - 2*x + 1, 1),
    (x**2 + 3*x - 1, 2),
    (sp.exp(x), 0),
    (sp.log(x), 1),
    (x**3 - 6*x, 2),
    (-x**2 + 4*x, 1),
    (sp.sin(x), 0),
    (x**4 - 2*x**2, 1),
    (sp.sqrt(x), 4),
    (1/x, 1),
]

def tangenten_quiz():
    f_expr, x0 = random.choice(tangenten_pool)
    fp = sp.diff(f_expr, x)
    y0 = f_expr.subs(x, x0)
    m = fp.subs(x, x0)
    # Tangente: t(x) = y0 + m*(x - x0), vereinfacht
    tangente = sp.expand(y0 + m * (x - x0))

    display(Markdown("### Bestimme die Tangentengleichung:"))
    display(Markdown(f"$$f(x) = {sp.latex(f_expr)} \\quad \\text{{im Punkt }} x_0 = {x0}$$"))

    eingabe = Text(
        placeholder='z.B. 2*x - 3',
        description='t(x) =',
        style={'description_width': '60px'},
        layout={'width': '500px'},
        continuous_update=False,
    )
    btn = Button(description='Prüfen', button_style='success')
    btn_loesung = Button(description='Lösung zeigen', button_style='warning')
    out = Output()

    def pruefen(b):
        with out:
            clear_output()
            try:
                antwort = parse_eingabe(eingabe.value)
                diff = sp.simplify(sp.expand(antwort) - tangente)
                if diff == 0:
                    display(Markdown("### Richtig!"))
                else:
                    display(Markdown("### Leider falsch."))
                    display(Markdown(f"Deine Antwort: $t(x) = {sp.latex(antwort)}$"))
                    display(Markdown(f"Richtig wäre: $t(x) = {sp.latex(tangente)}$"))
            except Exception as e:
                display(Markdown(f"**Eingabefehler:** {e}"))

    def zeige_loesung(b):
        with out:
            clear_output()
            display(Markdown("**Rechenweg:**"))
            display(Markdown(f"1. $f'(x) = {sp.latex(fp)}$"))
            display(Markdown(f"2. $f({x0}) = {sp.latex(y0)}$, $\\quad f'({x0}) = {sp.latex(m)}$"))
            display(Markdown(f"3. $t(x) = {sp.latex(y0)} + {sp.latex(m)} \\cdot (x - {x0})$"))
            display(Markdown(f"$$\\boxed{{t(x) = {sp.latex(tangente)}}}$$"))

    btn.on_click(pruefen)
    btn_loesung.on_click(zeige_loesung)
    display(VBox([eingabe, HBox([btn, btn_loesung]), out]))

tangenten_quiz()

---
## 7. Extrema Schritt für Schritt bestimmen

Hier gehst du das Standardverfahren durch:
1. $f'(x) = 0$ setzen und lösen
2. $f''(x_0)$ berechnen und Art bestimmen
3. Funktionswert berechnen

Jeder Schritt wird einzeln geprüft.

In [ ]:
# Aufgabenpool Extrema
extrema_pool = [
    x**3 - 6*x**2 + 9*x - 2,
    x**3 - 3*x,
    -x**3 + 3*x**2 - 1,
    x**4 - 4*x**2,
    x**3 + 3*x**2 - 9*x + 5,
    -2*x**3 + 6*x,
    x**4 - 8*x**2 + 16,
]

def extrema_trainer():
    f_expr = random.choice(extrema_pool)
    fp = sp.diff(f_expr, x)
    fpp = sp.diff(f_expr, x, 2)
    nullstellen = sp.solve(fp, x)
    # Nur reelle, rationale Nullstellen behalten
    nullstellen = sorted([n for n in nullstellen if n.is_real], key=float)

    display(Markdown("### Bestimme alle Extremstellen von:"))
    display(Markdown(f"$$f(x) = {sp.latex(f_expr)}$$"))

    # Schritt 1: Ableitung
    display(Markdown("**Schritt 1:** Bestimme $f'(x)$"))
    eingabe_fp = Text(placeholder="z.B. 3*x^2 - 12*x + 9", description="f'(x) =",
                      style={'description_width': '60px'}, layout={'width': '500px'},
                      continuous_update=False)
    btn1 = Button(description='Prüfen', button_style='success')
    out1 = Output()

    # Schritt 2: Nullstellen
    display_step2 = Output()
    # Schritt 3: Art + Funktionswerte
    display_step3 = Output()

    def schritt1(b):
        with out1:
            clear_output()
            try:
                antwort = parse_eingabe(eingabe_fp.value)
                if sp.simplify(antwort - fp) == 0:
                    display(Markdown("Richtig!"))
                    with display_step2:
                        clear_output()
                        zeige_schritt2()
                else:
                    display(Markdown(f"Nicht ganz. Richtig: $f'(x) = {sp.latex(sp.expand(fp))}$"))
                    with display_step2:
                        clear_output()
                        zeige_schritt2()
            except Exception as e:
                display(Markdown(f"**Eingabefehler:** {e}"))

    def zeige_schritt2():
        ns_str = ", ".join([f"$x = {sp.latex(n)}$" for n in nullstellen])
        display(Markdown(f"**Schritt 2:** Setze $f'(x) = 0$. Nullstellen: {ns_str}"))
        display(Markdown("**Schritt 3:** Bestimme die Art jeder Extremstelle mit $f''(x_0)$:"))
        display(Markdown(f"$f''(x) = {sp.latex(sp.expand(fpp))}$"))

        for x0 in nullstellen:
            fpp_wert = fpp.subs(x, x0)
            f_wert = f_expr.subs(x, x0)
            if fpp_wert > 0:
                art = "Tiefpunkt"
            elif fpp_wert < 0:
                art = "Hochpunkt"
            else:
                art = "keine Aussage (VZW von f' prüfen)"
            display(Markdown(
                f"- $x_0 = {sp.latex(x0)}$: "
                f"$f''({sp.latex(x0)}) = {sp.latex(fpp_wert)}$ "
                f"{'> 0' if fpp_wert > 0 else '< 0' if fpp_wert < 0 else '= 0'} "
                f"$\\Rightarrow$ **{art}**"
            ))
            display(Markdown(
                f"  Funktionswert: $f({sp.latex(x0)}) = {sp.latex(f_wert)}$ "
                f"$\\Rightarrow$ Punkt $({sp.latex(x0)} \\mid {sp.latex(f_wert)})$"
            ))

        # Graph zur Kontrolle
        xs_np = np.linspace(float(min(nullstellen)) - 2, float(max(nullstellen)) + 2, 300)
        f_np = sp.lambdify(x, f_expr, 'numpy')
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.plot(xs_np, f_np(xs_np), color='#1f77b4', linewidth=2.5,
                label=f'$f(x) = {sp.latex(f_expr)}$')
        for x0 in nullstellen:
            f_wert = float(f_expr.subs(x, x0))
            fpp_wert = fpp.subs(x, x0)
            farbe = '#d62728' if fpp_wert < 0 else '#2ca02c' if fpp_wert > 0 else '#ff7f0e'
            ax.plot(float(x0), f_wert, 'o', color=farbe, markersize=10, zorder=5)
            ax.annotate(f'$({sp.latex(x0)} \\mid {sp.latex(sp.Rational(f_wert).limit_denominator(100))})$',
                        (float(x0), f_wert), textcoords="offset points",
                        xytext=(10, 10), fontsize=11)
        ax.axhline(y=0, color='k', linewidth=0.5)
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=12)
        ax.set_xlabel('$x$')
        ax.set_ylabel('$y$')
        plt.tight_layout()
        plt.show()

    btn1.on_click(schritt1)
    display(VBox([eingabe_fp, btn1, out1, display_step2, display_step3]))

extrema_trainer()

In [ ]:
begruendungen = [
    {
        'frage': "**Aussage:** „$f'(x_0) = 0$ ist eine hinreichende Bedingung für ein Extremum bei $x_0$." — Wahr oder falsch?",
        'antwort': "**Falsch.** $f'(x_0) = 0$ ist nur eine *notwendige* Bedingung.\n\n"
                   "**Gegenbeispiel:** $f(x) = x^3$ hat $f'(0) = 0$, aber bei $x = 0$ liegt "
                   "kein Extremum vor (Sattelpunkt).\n\n"
                   "**Hinreichend** wäre z.B.: $f'(x_0) = 0$ **und** $f''(x_0) \\neq 0$.",
    },
    {
        'frage': "**Aussage:** „Wenn $f''(x_0) = 0$, dann hat $f$ bei $x_0$ einen Wendepunkt." — Wahr oder falsch?",
        'antwort': "**Falsch.** $f''(x_0) = 0$ ist nur *notwendig* für einen Wendepunkt.\n\n"
                   "**Gegenbeispiel:** $f(x) = x^4$ hat $f''(0) = 0$, aber $x = 0$ ist "
                   "kein Wendepunkt (Minimum).\n\n"
                   "**Hinreichend** wäre: $f''(x_0) = 0$ **und** Vorzeichenwechsel von $f''$ bei $x_0$.",
    },
    {
        'frage': "**Aussage:** „$f'(x_0) = 0$ und $f''(x_0) < 0$ — dann hat $f$ bei $x_0$ ein lokales Maximum." — Wahr oder falsch?",
        'antwort': "**Wahr.** Das ist die **hinreichende Bedingung** (2. Ableitung).\n\n"
                   "- $f'(x_0) = 0$: notwendige Bedingung erfüllt\n"
                   "- $f''(x_0) < 0$: Rechtskrümmung → Hochpunkt\n\n"
                   "Analog: $f''(x_0) > 0$ → Linkskrümmung → Tiefpunkt.",
    },
    {
        'frage': "**Aussage:** „Jede stetige Funktion hat ein Extremum." — Wahr oder falsch?",
        'antwort': "**Falsch** (auf ganz $\\mathbb{R}$). Zum Beispiel $f(x) = x$ ist stetig, "
                   "hat aber kein Extremum.\n\n"
                   "**Aber:** Auf einem *abgeschlossenen Intervall* $[a, b]$ hat jede stetige Funktion "
                   "ein globales Maximum und Minimum (Satz vom Maximum).",
    },
    {
        'frage': "**Aussage:** „Wenn $f$ bei $x_0$ ein lokales Maximum hat, dann gilt $f'(x_0) = 0$." — Wahr oder falsch?",
        'antwort': "**Wahr** (falls $f$ in $x_0$ differenzierbar ist). Das ist genau die "
                   "**notwendige Bedingung**.\n\n"
                   "An einer Extremstelle muss die Tangente waagerecht sein, also Steigung $= 0$.",
    },
    {
        'frage': "**Aufgabe:** Nenne alle Schritte, die du beim Bestimmen von Extrema durchführst.",
        'antwort': "**Standardverfahren Extrema:**\n\n"
                   "1. $f'(x) = 0$ setzen → Kandidaten $x_0$ bestimmen (*notwendige Bedingung*)\n"
                   "2. $f''(x_0)$ berechnen (*hinreichende Bedingung*):\n"
                   "   - $f''(x_0) > 0$ → **Tiefpunkt**\n"
                   "   - $f''(x_0) < 0$ → **Hochpunkt**\n"
                   "   - $f''(x_0) = 0$ → keine Aussage, VZW von $f'$ prüfen\n"
                   "3. Funktionswert $f(x_0)$ berechnen → Koordinaten des Extrempunkts",
    },
]

def begruendungs_trainer():
    aufgabe = random.choice(begruendungen)

    display(Markdown("### Begründungsaufgabe"))
    display(Markdown(aufgabe['frage']))

    # Textfeld für eigene Antwort
    display(Markdown("*Schreibe deine Begründung, bevor du die Lösung aufdeckst:*"))
    eigene_antwort = Textarea(
        placeholder='Deine Begründung hier eingeben...',
        layout={'width': '100%', 'height': '120px'},
    )
    btn = Button(description='Meine Antwort abgeben & Lösung zeigen', button_style='info')
    out = Output()

    def zeigen(b):
        with out:
            clear_output()
            if eigene_antwort.value.strip():
                display(Markdown("**Deine Antwort:**"))
                display(Markdown(f"> {eigene_antwort.value}"))
                display(Markdown("---"))
            display(Markdown("**Musterlösung:**"))
            display(Markdown(aufgabe['antwort']))
            if eigene_antwort.value.strip():
                display(Markdown("\n*Vergleiche deine Formulierung mit der Musterlösung. "
                                 "Hast du das Gegenbeispiel / die Begründung genannt?*"))

    btn.on_click(zeigen)
    display(VBox([eigene_antwort, btn, out]))

begruendungs_trainer()

---
## 9. Typische Fehler im Teil A — Stolperfallen vermeiden

Lies diese häufigen Fehler durch und präge sie dir ein. **Genau diese Fehler kosten im Abi Punkte!**

### Ableitungen

| Fehler | Falsch | Richtig |
|--------|--------|---------|
| Kettenregel vergessen | $(e^{-x^2})' = e^{-x^2}$ | $(e^{-x^2})' = -2x \cdot e^{-x^2}$ |
| Produktregel vergessen | $(x^2 \cdot \ln x)' = 2x \cdot \frac{1}{x}$ | $(x^2 \cdot \ln x)' = 2x \cdot \ln x + x$ |
| $e^x$ falsch ableiten | $(e^{3x})' = e^{3x}$ | $(e^{3x})' = 3 \cdot e^{3x}$ |
| $\ln(x)$ falsch ableiten | $(\ln(2x))' = \frac{1}{2x}$ | $(\ln(2x))' = \frac{2}{2x} = \frac{1}{x}$ |

### Integrale

| Fehler | Falsch | Richtig |
|--------|--------|---------|
| Integral = Fläche | $\int_0^2 (x^2-2x)\,dx = -\frac{4}{3}$ → "Fläche ist $-\frac{4}{3}$" | Fläche $= \left\|\int_0^2 (x^2-2x)\,dx\right\| = \frac{4}{3}$ |
| Stammfunktion von $e^{ax}$ | $\int e^{3x}\,dx = e^{3x}$ | $\int e^{3x}\,dx = \frac{1}{3}e^{3x}$ |
| Grenzen vertauscht | $F(a) - F(b)$ | $F(b) - F(a)$ (obere minus untere!) |

### Extrema & Bedingungen

| Fehler | Warum falsch? |
|--------|--------------|
| "$f'(x_0) = 0$ also Extremum" | Nur notwendig! Gegenbeispiel: $f(x) = x^3$ |
| "$f''(x_0) = 0$ also Wendepunkt" | Nur notwendig! Gegenbeispiel: $f(x) = x^4$ |
| Vorzeichen von $f''$ vertauscht | $f'' > 0$ → Linkskrümmung → **Tief**punkt (nicht Hoch!) |
| Funktionswert vergessen | Extremstelle allein reicht nicht — Extrempunkt hat $x$- UND $y$-Koordinate |

### Tangente

| Fehler | Richtig |
|--------|---------|
| $t(x) = f'(x_0) \cdot x$ | $t(x) = f(x_0) + f'(x_0) \cdot (x - x_0)$ |
| $f(x_0)$ und $f'(x_0)$ verwechselt | $f(x_0)$ = Funktionswert, $f'(x_0)$ = Steigung |

---
## Zusammenfassung: Checkliste für Teil A

Bevor du in die Prüfung gehst, solltest du das hier sicher können:

| Thema | Übung im Notebook | Sicher? |
|-------|-------------------|---------|
| Potenzregel, Summenregel | Abschnitt 2 | |
| Kettenregel | Abschnitt 2 | |
| Produktregel | Abschnitt 2 | |
| Ableitungen von $e^x$, $\ln(x)$, $\sin(x)$, $\cos(x)$ | Abschnitt 2 | |
| Stammfunktionen finden | Abschnitt 3 | |
| Bestimmtes Integral berechnen | Abschnitt 4 | |
| Integral vs. Fläche unterscheiden | Abschnitt 9 (Fehler) | |
| Graphen f, f', f'' zuordnen | Abschnitt 1 | |
| Notwendige vs. hinreichende Bedingung | Abschnitt 8 | |
| Tangentengleichung aufstellen | Abschnitt 6 | |
| Extrema bestimmen (ohne TR) | Abschnitt 7 | |
| Typische Fehler kennen | Abschnitt 9 | |

**Tipp:** Gehe die Abschnitte durch, bis du jede Zeile mit "sicher" abhaken kannst.

**Viel Erfolg im Abitur!**